# 📈 LSTM Time Series: Commodity Price Forecasting in East Africa

**Author:** Jok Akech Atem Mabior | CMU Africa

**Dataset:** Simulated weekly commodity price time series (2015–2024) for Maize, Beans, and Sorghum in East Africa.

---

This notebook builds an LSTM-based time series forecasting model for commodity prices, with direct applications to food security early warning systems across Kenya, Uganda, and Tanzania.


## Background: Commodity Price Volatility in East Africa

### The Food Price Crisis

Food price volatility is one of the leading drivers of food insecurity in East Africa:

- **Maize**: The primary staple crop for over 200 million people in East and Southern Africa. Prices are highly sensitive to rainfall variability, with a 30-50% price spike during drought years. In Kenya, maize accounts for ~36% of caloric intake.
- **Beans (Pulses)**: Critical protein source in low-income diets. Prices correlate with maize but have additional volatility from export markets to Europe and North America.
- **Sorghum**: Drought-tolerant staple critical in arid zones (northern Kenya, Ethiopia, Sudan border regions). More stable than maize but affected by conflict-related supply disruptions.

### Price Monitoring Systems

- **FAMEWS** (Famine Early Warning Systems Network): Collects market prices from 8,000+ markets across sub-Saharan Africa weekly
- **WFP VAM** (Vulnerability Analysis and Mapping): Monitors price trends for shock detection
- **KNBS** (Kenya National Bureau of Statistics): Monthly price indices

### Why LSTM?

Long Short-Term Memory networks are ideal for commodity price forecasting because:
1. **Long-range dependencies**: Harvest season effects manifest 3-6 months later as storage depletes
2. **Non-stationarity**: Prices trend upward with inflation while exhibiting seasonal patterns
3. **Shock memory**: Price shocks (drought, conflict, COVID-19) have persistent multi-week effects
4. **Complex seasonality**: Multiple overlapping seasonal patterns (short rains, long rains, local harvest calendars)

### Applications
- **Early Warning**: Predict price spikes 4-8 weeks in advance to trigger humanitarian pre-positioning
- **Trader Decision Support**: Help small-scale traders optimize buying/selling timing
- **Government Policy**: Inform strategic grain reserve releases to dampen price volatility


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8')

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
    from tensorflow.keras.callbacks import EarlyStopping
    print(f"TensorFlow: {tf.__version__}")
except ImportError:
    print("TensorFlow not available")
    tf = None

## Data Generation: East African Commodity Prices (2015–2024)

We simulate weekly price series with:
- **Trend**: Annual inflation ~2-2.5%
- **Seasonality**: Harvest season price dips (biannual in East Africa)
- **Random Walk**: Correlated noise representing market microstructure
- **Shock Events**: Drought/conflict spikes (3% probability per week, lasting 4-16 weeks)


In [ ]:
np.random.seed(42)
dates = pd.date_range(start='2015-01-01', end='2024-12-31', freq='W')  # weekly
n = len(dates)

def generate_price_series(base_price, trend=0.02, seasonality_amp=0.15,
                           shock_prob=0.03, shock_magnitude=0.25):
    """Generate realistic commodity price series with trend, seasonality, shocks."""
    t = np.arange(n)
    trend_component = base_price * (1 + trend/52 * t)
    # Annual seasonality (harvest season = lower prices)
    seasonal = seasonality_amp * np.sin(2 * np.pi * t / 52 - np.pi/2)
    # Random walk component
    noise = np.cumsum(np.random.normal(0, 0.008, n))
    # Price shocks (droughts, conflicts)
    shocks = np.zeros(n)
    shock_times = np.where(np.random.uniform(0, 1, n) < shock_prob)[0]
    for st in shock_times:
        duration = np.random.randint(4, 16)
        shock_size = np.random.uniform(0.15, shock_magnitude)
        shocks[st:min(st+duration, n)] += shock_size
    price = trend_component * (1 + seasonal + noise + shocks)
    return price.clip(base_price * 0.4, base_price * 3.5)

# Generate prices for three commodities (KES per 90kg bag)
df = pd.DataFrame({
    'date': dates,
    'maize_kes': generate_price_series(2800, trend=0.025, seasonality_amp=0.18),
    'beans_kes': generate_price_series(7500, trend=0.020, seasonality_amp=0.12),
    'sorghum_kes': generate_price_series(2200, trend=0.018, seasonality_amp=0.14),
})

# Add moving averages and returns
for col in ['maize_kes', 'beans_kes', 'sorghum_kes']:
    df[f'{col}_ma4']  = df[col].rolling(4).mean()
    df[f'{col}_ma12'] = df[col].rolling(12).mean()
    df[f'{col}_pct_change'] = df[col].pct_change()

df = df.dropna().reset_index(drop=True)
print(f"Time series dataset: {df.shape}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"\nPrice statistics:")
print(df[['maize_kes','beans_kes','sorghum_kes']].describe().round(2))

## EDA 1: Commodity Price Time Series


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
commodities = [('maize_kes', 'Maize', '#F4A460'), ('beans_kes', 'Beans', '#8B4513'), ('sorghum_kes', 'Sorghum', '#DAA520')]

for i, (col, name, color) in enumerate(commodities):
    axes[i].plot(df['date'], df[col], color=color, linewidth=1, label='Weekly Price')
    axes[i].plot(df['date'], df[f'{col}_ma12'], 'r--', linewidth=1.5, label='12-week MA')
    axes[i].fill_between(df['date'], df[col], alpha=0.2, color=color)
    axes[i].set_ylabel(f'KES per 90kg', fontsize=10)
    axes[i].set_title(f'{name} Price — East Africa (2015-2024)', fontweight='bold')
    axes[i].legend(fontsize=9)
    axes[i].grid(True, alpha=0.3)

axes[2].set_xlabel('Date')
plt.suptitle('East African Commodity Price Trends', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## EDA 2: Price Distributions and Seasonality


In [ ]:
df['month'] = pd.to_datetime(df['date']).dt.month
df['year']  = pd.to_datetime(df['date']).dt.year

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Price distributions
for col, name, color in commodities:
    axes[0][0].hist(df[col], bins=40, alpha=0.5, color=color, label=name, density=True)
axes[0][0].set_title('Price Distributions (2015-2024)', fontweight='bold')
axes[0][0].set_xlabel('KES per 90kg'); axes[0][0].legend()

# Correlation between commodities
corr = df[['maize_kes', 'beans_kes', 'sorghum_kes']].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', ax=axes[0][1],
            xticklabels=['Maize','Beans','Sorghum'], yticklabels=['Maize','Beans','Sorghum'])
axes[0][1].set_title('Commodity Price Correlation', fontweight='bold')

# Seasonal pattern — maize by month
maize_monthly = df.groupby('month')['maize_kes'].mean()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1][0].bar(month_names, maize_monthly.values, color='#F4A460', alpha=0.85, edgecolor='white')
axes[1][0].set_title('Avg Maize Price by Month (Seasonal Pattern)', fontweight='bold')
axes[1][0].set_xlabel('Month'); axes[1][0].set_ylabel('KES per 90kg')
axes[1][0].tick_params(axis='x', rotation=30)

# Year-over-year price trend
for col, name, color in commodities:
    yearly = df.groupby('year')[col].mean()
    axes[1][1].plot(yearly.index, yearly.values, '-o', color=color, label=name, lw=2)
axes[1][1].set_title('Annual Average Prices', fontweight='bold')
axes[1][1].set_xlabel('Year'); axes[1][1].set_ylabel('KES per 90kg')
axes[1][1].legend()

plt.suptitle('Commodity Price EDA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## EDA 3: Price Volatility and Returns Analysis


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (col, name, color) in enumerate(commodities):
    pct_col = f'{col}_pct_change'
    axes[idx].plot(df['date'], df[pct_col], color=color, linewidth=0.8, alpha=0.8)
    axes[idx].axhline(0, color='black', linewidth=0.5)
    axes[idx].axhline(df[pct_col].mean() + 2*df[pct_col].std(), color='red', ls='--',
                       linewidth=1, label='+2σ shock threshold')
    axes[idx].set_title(f'{name} Weekly Returns', fontweight='bold')
    axes[idx].set_xlabel('Date')
    axes[idx].set_ylabel('% Change')
    axes[idx].legend(fontsize=8)

    vol = df[pct_col].std() * 100
    print(f"{name} weekly volatility: {vol:.2f}%")

plt.suptitle('Weekly Price Returns and Shock Identification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## LSTM Data Preparation

We create sliding window sequences for multi-step forecasting:
- **Lookback**: 12 weeks of history
- **Horizon**: 4 weeks ahead forecast


In [ ]:
# Focus on maize price forecasting
price_series = df['maize_kes'].values.reshape(-1, 1)
scaler = MinMaxScaler(feature_range=(0, 1))
price_scaled = scaler.fit_transform(price_series)

# Create sequences
def create_sequences(data, lookback=12, horizon=4):
    """Create input/output sequences for LSTM."""
    X_seq, y_seq = [], []
    for i in range(lookback, len(data) - horizon + 1):
        X_seq.append(data[i-lookback:i, 0])
        y_seq.append(data[i:i+horizon, 0])
    return np.array(X_seq), np.array(y_seq)

LOOKBACK = 12   # 12 weeks of history
HORIZON  = 4    # predict 4 weeks ahead

X_seq, y_seq = create_sequences(price_scaled, LOOKBACK, HORIZON)
X_seq = X_seq.reshape(X_seq.shape[0], X_seq.shape[1], 1)

# Chronological train/test split (80/20)
split = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")
print(f"Total sequences: {len(X_seq)}")
print(f"Train period: {df['date'].iloc[LOOKBACK].date()} to {df['date'].iloc[split+LOOKBACK].date()}")
print(f"Test period:  {df['date'].iloc[split+LOOKBACK].date()} to {df['date'].iloc[-HORIZON].date()}")

## LSTM Model Architecture

A 3-layer stacked LSTM with Dropout and Huber loss (robust to outliers/price shocks):


In [ ]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(LOOKBACK, 1)),
    Dropout(0.2),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model.compile(optimizer='adam', loss='huber', metrics=['mae'])
model.summary()

## Model Training


In [ ]:
callbacks = [EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)]

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=1
)

plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('Huber Loss')
plt.title('LSTM Training History — Maize Price Forecasting')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Training stopped at epoch: {len(history.history['loss'])}")
print(f"Best validation loss: {min(history.history['val_loss']):.6f}")

## Evaluation and Forecast Plot


In [ ]:
y_pred_scaled = model.predict(X_test)

# Inverse transform (take mean of multi-step forecast)
y_test_inv = scaler.inverse_transform(y_test)
y_pred_inv = scaler.inverse_transform(y_pred_scaled)

# Use 1-step-ahead for primary metrics
rmse = np.sqrt(mean_squared_error(y_test_inv[:, 0], y_pred_inv[:, 0]))
mae  = mean_absolute_error(y_test_inv[:, 0], y_pred_inv[:, 0])
r2   = r2_score(y_test_inv[:, 0], y_pred_inv[:, 0])

print(f"1-Step-Ahead Forecast Metrics:")
print(f"  RMSE: {rmse:.2f} KES")
print(f"  MAE:  {mae:.2f} KES")
print(f"  R2:   {r2:.4f}")
print(f"  MAPE: {np.mean(np.abs((y_test_inv[:,0] - y_pred_inv[:,0]) / y_test_inv[:,0]))*100:.2f}%")

# Plot forecast vs actual
test_dates = df['date'].values[split + LOOKBACK + HORIZON - 1: split + LOOKBACK + HORIZON - 1 + len(y_test)]
plt.figure(figsize=(14, 5))
plt.plot(test_dates[:len(y_test_inv)], y_test_inv[:, 0], label='Actual Price', color='blue', lw=1.5)
plt.plot(test_dates[:len(y_pred_inv)], y_pred_inv[:, 0], label='LSTM Forecast', color='red', lw=1.5, ls='--')
plt.xlabel('Date'); plt.ylabel('Maize Price (KES/90kg)')
plt.title(f'LSTM Maize Price Forecast vs Actual (RMSE={rmse:.0f} KES)', fontweight='bold')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Multi-Step Forecast Horizon Analysis


In [ ]:
horizon_metrics = []
for h in range(HORIZON):
    rmse_h = np.sqrt(mean_squared_error(y_test_inv[:, h], y_pred_inv[:, h]))
    mae_h  = mean_absolute_error(y_test_inv[:, h], y_pred_inv[:, h])
    horizon_metrics.append({'Horizon (weeks)': h+1, 'RMSE': rmse_h, 'MAE': mae_h})

hm_df = pd.DataFrame(horizon_metrics)
print("Forecast accuracy by horizon:")
print(hm_df.round(2).to_string(index=False))

plt.figure(figsize=(10, 4))
plt.plot(hm_df['Horizon (weeks)'], hm_df['RMSE'], 'r-o', lw=2, label='RMSE')
plt.plot(hm_df['Horizon (weeks)'], hm_df['MAE'],  'b-o', lw=2, label='MAE')
plt.xlabel('Forecast Horizon (weeks)')
plt.ylabel('Error (KES)')
plt.title('Forecast Error by Horizon', fontweight='bold')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Residual Analysis


In [ ]:
residuals = y_test_inv[:, 0] - y_pred_inv[:, 0]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Residuals over time
axes[0].plot(test_dates[:len(residuals)], residuals, color='purple', lw=1)
axes[0].axhline(0, color='black', lw=1)
axes[0].axhline(residuals.mean() + 2*residuals.std(), color='red', ls='--', label='+2σ')
axes[0].axhline(residuals.mean() - 2*residuals.std(), color='red', ls='--', label='-2σ')
axes[0].set_title('Residuals Over Time', fontweight='bold')
axes[0].set_xlabel('Date'); axes[0].set_ylabel('Residual (KES)')
axes[0].legend()

# Residual histogram
axes[1].hist(residuals, bins=40, color='steelblue', alpha=0.8, edgecolor='white', density=True)
from scipy import stats
x_range = np.linspace(residuals.min(), residuals.max(), 100)
axes[1].plot(x_range, stats.norm.pdf(x_range, residuals.mean(), residuals.std()),
             'r-', lw=2, label='Normal fit')
axes[1].set_title(f'Residual Distribution (mean={residuals.mean():.1f}, std={residuals.std():.1f})',
                  fontweight='bold')
axes[1].set_xlabel('Residual (KES)'); axes[1].legend()

# Actual vs Predicted scatter
axes[2].scatter(y_test_inv[:, 0], y_pred_inv[:, 0], alpha=0.3, s=15, color='steelblue')
lims = [min(y_test_inv[:, 0].min(), y_pred_inv[:, 0].min()),
        max(y_test_inv[:, 0].max(), y_pred_inv[:, 0].max())]
axes[2].plot(lims, lims, 'r--', lw=2, label='Perfect forecast')
axes[2].set_xlabel('Actual Price (KES)')
axes[2].set_ylabel('Predicted Price (KES)')
axes[2].set_title(f'Actual vs Predicted (R²={r2:.4f})', fontweight='bold')
axes[2].legend()

plt.suptitle('LSTM Residual Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Multi-Commodity Forecasting Comparison


In [ ]:
commodity_metrics = []
for col, name, color in [('maize_kes', 'Maize', '#F4A460'), 
                           ('beans_kes', 'Beans', '#8B4513'), 
                           ('sorghum_kes', 'Sorghum', '#DAA520')]:
    ps = df[col].values.reshape(-1, 1)
    sc = MinMaxScaler().fit(ps)
    ps_scaled = sc.fit_transform(ps)

    Xs, ys = create_sequences(ps_scaled, LOOKBACK, HORIZON)
    Xs = Xs.reshape(Xs.shape[0], Xs.shape[1], 1)
    sp = int(0.8 * len(Xs))
    Xtr_c, Xte_c = Xs[:sp], Xs[sp:]
    ytr_c, yte_c = ys[:sp], ys[sp:]

    m = Sequential([
        LSTM(32, return_sequences=True, input_shape=(LOOKBACK, 1)),
        Dropout(0.2),
        LSTM(16),
        Dense(HORIZON)
    ])
    m.compile(optimizer='adam', loss='mse')
    m.fit(Xtr_c, ytr_c, epochs=30, batch_size=16, verbose=0,
          callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])

    ypred_c = m.predict(Xte_c, verbose=0)
    yte_inv = sc.inverse_transform(yte_c)
    ypred_inv = sc.inverse_transform(ypred_c)

    rmse_c = np.sqrt(mean_squared_error(yte_inv[:,0], ypred_inv[:,0]))
    mape_c = np.mean(np.abs((yte_inv[:,0] - ypred_inv[:,0]) / yte_inv[:,0])) * 100
    r2_c   = r2_score(yte_inv[:,0], ypred_inv[:,0])

    commodity_metrics.append({'Commodity': name, 'RMSE (KES)': rmse_c,
                               'MAPE (%)': mape_c, 'R2': r2_c})
    print(f"{name}: RMSE={rmse_c:.1f} KES, MAPE={mape_c:.2f}%, R2={r2_c:.4f}")

cdf = pd.DataFrame(commodity_metrics)
print("\nCommodity Forecast Comparison:")
print(cdf.round(3).to_string(index=False))

## Forecast Visualization: Future 4-Week Prediction


In [ ]:
# Use last available data point to forecast the next 4 weeks
last_window = price_scaled[-LOOKBACK:].reshape(1, LOOKBACK, 1)
future_pred_scaled = model.predict(last_window, verbose=0)
future_pred_kes = scaler.inverse_transform(future_pred_scaled)[0]

last_date = df['date'].iloc[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(weeks=1),
                              periods=HORIZON, freq='W')

# Plot last 26 weeks + 4-week forecast
hist_window = 26
recent_dates = df['date'].iloc[-hist_window:]
recent_prices = df['maize_kes'].iloc[-hist_window:]

plt.figure(figsize=(14, 6))
plt.plot(recent_dates, recent_prices, 'b-o', lw=2, ms=4, label='Historical (last 26 weeks)')
plt.plot([recent_dates.iloc[-1]] + list(future_dates),
         [recent_prices.iloc[-1]] + list(future_pred_kes),
         'r--o', lw=2, ms=8, label='4-Week Forecast', color='red')
plt.fill_between(future_dates,
                  future_pred_kes * 0.95,
                  future_pred_kes * 1.05,
                  alpha=0.2, color='red', label='±5% Confidence Interval')
plt.axvline(last_date, color='gray', ls='--', label='Forecast Start')
plt.xlabel('Date'); plt.ylabel('Maize Price (KES/90kg)')
plt.title('Maize Price Forecast: Next 4 Weeks', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n4-Week Forward Forecast:")
for i, (d, p) in enumerate(zip(future_dates, future_pred_kes)):
    print(f"  Week {i+1} ({d.date()}): {p:.0f} KES/90kg")

## Naive Baseline Comparison


In [ ]:
# Naive baseline: predict last observed value (random walk)
y_naive = y_test_inv[:, 0].copy()
# Shift: use last input value as forecast
last_inputs_scaled = X_test[:, -1, 0]  # last value in each input window
last_inputs_kes = scaler.inverse_transform(last_inputs_scaled.reshape(-1, 1)).flatten()

rmse_naive = np.sqrt(mean_squared_error(y_test_inv[:, 0], last_inputs_kes))
mae_naive  = mean_absolute_error(y_test_inv[:, 0], last_inputs_kes)
mape_naive = np.mean(np.abs((y_test_inv[:,0] - last_inputs_kes) / y_test_inv[:,0])) * 100

print("Model Comparison (1-step ahead):")
print(f"{'Model':<20} {'RMSE':>8} {'MAE':>8} {'MAPE%':>8}")
print(f"{'LSTM':<20} {rmse:>8.2f} {mae:>8.2f} {np.mean(np.abs((y_test_inv[:,0] - y_pred_inv[:,0]) / y_test_inv[:,0]))*100:>8.2f}")
print(f"{'Naive (last value)':<20} {rmse_naive:>8.2f} {mae_naive:>8.2f} {mape_naive:>8.2f}")

models_comp = ['Naive Baseline', 'LSTM (3-layer)']
rmses_comp  = [rmse_naive, rmse]

plt.figure(figsize=(8, 5))
bars = plt.bar(models_comp, rmses_comp, color=['#95a5a6', '#e74c3c'], alpha=0.85, edgecolor='white')
plt.title('LSTM vs Naive Baseline (RMSE)', fontweight='bold')
plt.ylabel('RMSE (KES per 90kg)')
for bar, v in zip(bars, rmses_comp):
    plt.text(bar.get_x() + bar.get_width()/2, v + 5, f'{v:.1f}', ha='center', fontsize=12)
plt.tight_layout()
plt.show()

improvement = (rmse_naive - rmse) / rmse_naive * 100
print(f"\nLSTM improves over naive baseline by {improvement:.1f}% in RMSE")

## Conclusions

### LSTM Performance

The 3-layer stacked LSTM with Huber loss successfully captured the temporal patterns in East African commodity price data:

| Metric | 1-Week | 2-Week | 3-Week | 4-Week |
|--------|--------|--------|--------|--------|
| RMSE | Lowest | Increasing | Increasing | Highest |
| MAE | Lowest | Increasing | Increasing | Highest |

As expected, forecast error increases with horizon — the model is most reliable for 1-2 week ahead predictions.

### Use for Food Security Early Warning

1. **4-week ahead spike detection**: When forecast exceeds +20% of 12-week moving average → trigger early warning alert
2. **Regional aggregation**: Run separate models per market cluster, aggregate to county/district level
3. **WFP integration**: LSTM outputs could feed into FEWS NET alerts alongside rainfall and crop condition data
4. **Trader apps**: SMS-based weekly price forecasts for traders in markets with limited connectivity

### Limitations

- **External shocks are hard to predict**: The 2020 COVID-19 shock, 2022 Russia-Ukraine war (fertilizer prices), and locust invasions are not predictable from historical price patterns alone
- **Market integration**: Prices in isolated markets may not follow the same patterns as hub markets
- **Data quality**: FAMEWS data has gaps, reporting delays, and potential measurement error
- **Model drift**: Structural breaks (new roads, improved storage) require periodic retraining

### Next Steps

1. **Multivariate LSTM**: Add rainfall anomalies (CHIRPS), NDVI vegetation index, and conflict events (ACLED) as exogenous features
2. **Attention Mechanisms / Transformer**: Self-attention can capture long-range dependencies more efficiently than stacked LSTMs
3. **Probabilistic Forecasting**: Output prediction intervals (quantile LSTM) rather than point estimates
4. **Transfer Learning**: Pre-train on data-rich markets, fine-tune on data-scarce rural markets
5. **Ensemble Methods**: Combine LSTM with ARIMA and gradient boosting for improved robustness
